# P2 - RNNs

**Authors**:
- Pablo Diaz 
- Daniel Prieto

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Load the data and preprocess it

In [2]:
df = pd.read_csv("data/nyiso_hourly_load.csv")
df.head()

,Time Stamp,CAPITL,CENTRL,DUNWOD,GENESE,HUD VL,LONGIL,MHK VL,MILLWD,N.Y.C.,NORTH,WEST,total_load
0,2021-01-01 05:00:00+00:00,1229.765829,1652.064229,601.434650,980.770850,1001.962714,2061.066821,841.068829,319.410071,4808.552936,635.814164,1525.174400,15657.085493
1,2021-01-01 06:00:00+00:00,1199.764783,1606.512858,579.089583,934.267075,960.858267,1946.459783,807.607275,307.969050,4631.978808,633.340358,1473.827217,15081.675058
2,2021-01-01 07:00:00+00:00,1172.361075,1578.671450,559.861700,913.383017,936.291717,1855.849567,796.252242,301.264983,4486.644667,617.937925,1435.501408,14654.019750
3,2021-01-01 08:00:00+00:00,1164.461967,1556.862842,548.799792,901.166967,916.932750,1797.270625,794.205750,295.269642,4372.209042,615.713533,1417.398217,14380.291125
4,2021-01-01 09:00:00+00:00,1177.303475,1561.628092,545.171408,898.999517,915.355317,1791.221000,798.552250,295.507442,4336.574433,615.466042,1419.729483,14355.508458


In [3]:
print(df.columns)

Index(['Time Stamp', 'CAPITL', 'CENTRL', 'DUNWOD', 'GENESE', 'HUD VL',
       'LONGIL', 'MHK VL', 'MILLWD', 'N.Y.C.', 'NORTH', 'WEST', 'total_load'],
      dtype='str')


Temporal Split 

- Training set: 2021–2023
- Validation set: 2024
- Test set: 2025

In [4]:
# split the data into training/validation/test sets

# Training Set 2021 to 2023
train_df = df[df["Time Stamp"] < "2024-01-01"]
print(len(train_df))

# Validation Set 2024
val_df = df[(df["Time Stamp"] >= "2024-01-01") & (df["Time Stamp"] < "2025-01-01")]
print(len(val_df))

# Test Set 2025
test_df = df[df["Time Stamp"] >= "2025-01-01"]
print(len(test_df))

# drop the "Time Stamp" column
train_df = train_df.drop(columns=["Time Stamp"])
val_df = val_df.drop(columns=["Time Stamp"])
test_df = test_df.drop(columns=["Time Stamp"])

26107
8784
8765


In [5]:
mean = train_df.mean()
std = train_df.std()

print(f"Mean:\n{mean}\n")
print(f"Standard Deviation:\n{std}\n")

Mean:
CAPITL         1327.515791
CENTRL         1736.799880
DUNWOD          646.373512
GENESE         1087.183369
HUD VL         1051.645915
LONGIL         2276.543855
MHK VL          837.988563
MILLWD          322.531092
N.Y.C.         5593.559854
NORTH           646.981059
WEST           1676.361131
total_load    17203.484021
dtype: float64

Standard Deviation:
CAPITL         233.315696
CENTRL         272.144257
DUNWOD         146.049529
GENESE         189.547436
HUD VL         232.391308
LONGIL         634.965917
MHK VL         159.596627
MILLWD          76.722991
N.Y.C.        1148.912783
NORTH           65.901310
WEST           224.922815
total_load    3085.204590
dtype: float64



In [8]:
train_df_scaled = (train_df - mean) / std
val_df_scaled = (val_df - mean) / std
test_df_scaled = (test_df - mean) / std

In [ ]:
from keras.utils import timeseries_dataset_from_array

# we will create sequences of 24 hours to predict three hours in the future
sequence_length = 24
batch_size = 32
sample_rate = 1  # hourly data, so one sample per hour
delay = sample_rate * (sequence_length + 3 - 1)  # 3 hours in the future

train_ds = timeseries_dataset_from_array(
    data=train_df_scaled.values,
    targets=train_df_scaled["total_load"][delay:].values,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

val_ds = timeseries_dataset_from_array(
    data=val_df_scaled.values,
    targets=val_df_scaled["total_load"][delay:].values,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

test_ds = timeseries_dataset_from_array(
    data=test_df_scaled.values,
    targets=test_df_scaled["total_load"][delay:].values,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

NumpyIterator(iterator=<tensorflow.python.data.ops.iterator_ops.OwnedIterator object at 0x303a0c1a0>)

## Baselines

### Last Value Baseline



In [18]:
def evaluate_naive(dataset):
    total_abs_error = 0
    samples_counter = 0

    for samples, targets in dataset:
        print(samples, targets)
        predictions = samples[:, -1, -1] * std[-1] + mean[-1]
        total_abs_error += np.sum(np.abs(predictions - targets))
        samples_counter += samples_counter + samples.shape[0]


print(f"Validation MAE: {evaluate_naive(val_ds):.2f}")
print(f"Test MAE: {evaluate_naive(test_ds):.2f}")

tf.Tensor(
[[[ 0.38403778  0.56275912  0.13120797 ...  1.57557261  0.57218404
    0.29670505]
  [ 0.17128684  0.34688607  0.06175511 ...  1.50814654  0.44053558
    0.12402105]
  [-0.0687508   0.16072176 -0.07399137 ...  1.37644898  0.25911253
   -0.05954028]
  ...
  [ 0.48623797  0.95475045 -0.21775281 ...  1.55744867  0.74997547
    0.22411808]
  [ 0.92501288  1.23798082  0.10591581 ...  1.70589197  1.06048755
    0.52475365]
  [ 0.90101118  1.21785583  0.14825487 ...  1.55737242  1.0487762
    0.51881641]]

 [[ 0.17128684  0.34688607  0.06175511 ...  1.50814654  0.44053558
    0.12402105]
  [-0.0687508   0.16072176 -0.07399137 ...  1.37644898  0.25911253
   -0.05954028]
  [-0.28604819 -0.11516786 -0.29363084 ...  1.27968952  0.06209739
   -0.25677331]
  ...
  [ 0.92501288  1.23798082  0.10591581 ...  1.70589197  1.06048755
    0.52475365]
  [ 0.90101118  1.21785583  0.14825487 ...  1.55737242  1.0487762
    0.51881641]
  [ 0.78410613  1.10771354  0.08284658 ...  1.5465809   0.942808

KeyError: -1

### Last Day Baseline



### Neural Baseline with 1D Conv 

